# 📊 Stock Recommender — Step 1: Dataset EDA & Audit

**Goal of this step:** Understand the dataset deeply before touching any model.
Every design decision downstream (label thresholds, price fetch windows, dedup strategy, 
feature choices) depends on what we discover here.

**Iterative workflow:**
```
Step 1 → EDA & Audit            ← YOU ARE HERE
Step 2 → Price Data Fetch       ← next (only after reviewing Step 1 output)
Step 3 → Price-Reaction Labels  ← after Step 2
Step 4 → NLP Preprocessing      ← after Step 3
Step 5 → FinBERT Features       ← after Step 4
Step 6 → Model Training         ← after Step 5
Step 7 → Evaluation & Tune      ← after Step 6
```

> ⚠️ **Review every plot and printed audit block carefully before running Step 2.**
> Flag anything unexpected — it will affect label quality.


## ⚙️ 0. Install & Imports

In [ ]:
# Install only what Step 1 needs — lightweight
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from collections import Counter
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 80)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
sns.set_palette("husl")

print("✅ Imports ready.")


## 📂 1. Load Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload news_raw.csv via Colab:
#   from google.colab import files; files.upload()
# OR mount Drive:
#   from google.colab import drive; drive.mount('/content/drive')
#   CSV_PATH = '/content/drive/MyDrive/news_raw.csv'
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH = "news_raw.csv"   # ← adjust if using Drive

df_raw = pd.read_csv(CSV_PATH)

print(f"Shape        : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns      : {df_raw.columns.tolist()}")
print()
display(df_raw.head(4))


## 🔍 2. Schema, Data Types & Missing Values

In [ ]:
# ── Schema overview ────────────────────────────────────────────────────────────
print("=" * 65)
print(f"{'Column':<22} {'Dtype':<12} {'Non-Null':>9} {'Null':>9} {'Null%':>7}")
print("=" * 65)
for col in df_raw.columns:
    nn   = df_raw[col].notna().sum()
    null = df_raw[col].isna().sum()
    pct  = null / len(df_raw) * 100
    dtype = str(df_raw[col].dtype)
    flag = " ⚠️" if pct > 0 else ""
    print(f"  {col:<20} {dtype:<12} {nn:>9,} {null:>9,} {pct:>6.1f}%{flag}")
print("=" * 65)

print()
print("── Numeric columns summary ──────────────────────────────────────")
display(df_raw.describe())


In [ ]:
# ── Key finding: label columns are 100% null ──────────────────────────────────
label_cols = ['sentiment_pos', 'sentiment_neu', 'sentiment_neg', 'sentiment_label']
null_pct = {c: df_raw[c].isna().mean()*100 for c in label_cols}

print("LABEL COLUMN NULL RATES:")
for c, pct in null_pct.items():
    status = "🔴 COMPLETELY EMPTY" if pct == 100 else f"🟡 {pct:.1f}% null"
    print(f"  {c:<22}: {status}")

print()
print("💡 IMPLICATION: We have ZERO ready-made labels.")
print("   → We will generate labels using price reactions (Step 3)")
print("     which is significantly stronger than sentiment-only labels.")


## 📅 3. Temporal Distribution (Critical for Price Fetch Planning)

In [ ]:
# ── Parse all timestamps ───────────────────────────────────────────────────────
df_raw['pub_dt'] = pd.to_datetime(df_raw['published_at'], utc=True, errors='coerce')
df_raw['pub_year']      = df_raw['pub_dt'].dt.year
df_raw['pub_month']     = df_raw['pub_dt'].dt.month
df_raw['pub_yearmonth'] = df_raw['pub_dt'].dt.to_period('M')
df_raw['pub_date']      = df_raw['pub_dt'].dt.date
df_raw['pub_hour']      = df_raw['pub_dt'].dt.hour
df_raw['pub_dow']       = df_raw['pub_dt'].dt.dayofweek   # 0=Mon

invalid_dt = df_raw['pub_dt'].isna().sum()
print(f"Invalid/unparseable timestamps : {invalid_dt}")
print(f"Date range : {df_raw['pub_dt'].min()} → {df_raw['pub_dt'].max()}")
print()

year_counts = df_raw['pub_year'].value_counts().sort_index()
print("Year distribution:")
for yr, cnt in year_counts.items():
    bar = "█" * (cnt // 50)
    print(f"  {yr}  {cnt:5,}  {bar}")


In [ ]:
# ── Visualise temporal spread ──────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Articles per year (log-scale to show the long tail)
ax1 = fig.add_subplot(gs[0, 0])
yc = df_raw['pub_year'].value_counts().sort_index()
colors_yr = ['#F44336' if yr < 2023 else '#FF9800' if yr == 2023 else '#4CAF50'
             for yr in yc.index]
ax1.bar(yc.index.astype(str), yc.values, color=colors_yr, edgecolor='white', linewidth=1)
ax1.set_title("Articles per Year\n🔴pre-2023  🟠2023  🟢2024-26", fontweight='bold')
ax1.set_xlabel("Year"); ax1.set_ylabel("Count")
ax1.tick_params(axis='x', rotation=45)
for i, (yr, v) in enumerate(yc.items()):
    ax1.text(i, v + 50, str(v), ha='center', fontsize=8, rotation=45)

# 2. Monthly trend (2025-2026 only)
ax2 = fig.add_subplot(gs[0, 1])
recent = df_raw[df_raw['pub_year'] >= 2025].copy()
monthly = recent.groupby('pub_yearmonth').size()
ax2.plot(monthly.index.astype(str), monthly.values,
         marker='o', color='steelblue', linewidth=2, markersize=5)
ax2.fill_between(range(len(monthly)), monthly.values, alpha=0.2, color='steelblue')
ax2.set_title("Monthly Volume (2025–2026)", fontweight='bold')
ax2.set_xlabel("Month"); ax2.set_ylabel("Articles")
ax2.tick_params(axis='x', rotation=45)

# 3. Hour of day
ax3 = fig.add_subplot(gs[0, 2])
hour_c = df_raw['pub_hour'].value_counts().sort_index()
bar_colors = ['#FF9800' if 4 <= h <= 9 else '#2196F3' if 9 < h <= 16
              else '#9C27B0' if 16 < h <= 20 else '#9E9E9E' for h in hour_c.index]
ax3.bar(hour_c.index, hour_c.values, color=bar_colors, edgecolor='white')
ax3.set_title("Articles by Hour (UTC)\n🟠pre-mkt  🔵market  🟣after-hrs", fontweight='bold')
ax3.set_xlabel("Hour (UTC)"); ax3.set_ylabel("Count")
ax3.axvline(9.5,  color='orange', linestyle='--', linewidth=1.2, alpha=0.7)
ax3.axvline(16,   color='blue',   linestyle='--', linewidth=1.2, alpha=0.7)

# 4. Day of week
ax4 = fig.add_subplot(gs[1, 0])
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_c = df_raw['pub_dow'].value_counts().sort_index()
dow_colors = ['#4CAF50']*5 + ['#F44336']*2
ax4.bar([dow_labels[i] for i in dow_c.index], dow_c.values,
        color=[dow_colors[i] for i in dow_c.index], edgecolor='white')
ax4.set_title("Articles by Day of Week\n🟢Weekday  🔴Weekend", fontweight='bold')
ax4.set_xlabel("Day"); ax4.set_ylabel("Count")

# 5. Region split
ax5 = fig.add_subplot(gs[1, 1])
region_c = df_raw['region'].value_counts()
wedge_colors = ['#2196F3', '#4CAF50', '#FF5722']
ax5.pie(region_c.values, labels=region_c.index, autopct='%1.1f%%',
        colors=wedge_colors, startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax5.set_title("Market Region Split", fontweight='bold')

# 6. Pre-2023 outlier tickers
ax6 = fig.add_subplot(gs[1, 2])
outliers = df_raw[df_raw['pub_year'] < 2023].groupby('ticker').size().sort_values()
if len(outliers) > 0:
    ax6.barh(outliers.index, outliers.values, color='#F44336', alpha=0.8)
    ax6.set_title(f"Pre-2023 Articles by Ticker\n(total: {outliers.sum()} rows — will be flagged)",
                  fontweight='bold')
    ax6.set_xlabel("Count")
else:
    ax6.text(0.5, 0.5, 'No pre-2023 data', ha='center', va='center', fontsize=12)

plt.suptitle("Temporal Distribution Analysis", fontsize=15, fontweight='bold')
plt.savefig("eda_temporal.png", bbox_inches='tight', dpi=150)
plt.show()
print("\n💡 KEY INSIGHT: ~97% of data is 2025-2026 → yfinance can fetch price data for almost all rows.")


## 📈 4. Ticker & Market Segment Analysis

In [ ]:
# ── Per-ticker statistics ──────────────────────────────────────────────────────
ticker_stats = df_raw.groupby(['ticker', 'region']).agg(
    count       = ('title', 'count'),
    unique_days = ('pub_date', 'nunique'),
    date_min    = ('pub_dt', 'min'),
    date_max    = ('pub_dt', 'max'),
).reset_index()

ticker_stats['days_span'] = (
    pd.to_datetime(ticker_stats['date_max']) -
    pd.to_datetime(ticker_stats['date_min'])
).dt.days

ticker_stats['articles_per_day'] = (
    ticker_stats['count'] / ticker_stats['unique_days'].clip(lower=1)
).round(2)

# Pre-2023 count per ticker
early = df_raw[df_raw['pub_year'] < 2023].groupby('ticker').size().rename('pre2023_count')
ticker_stats = ticker_stats.merge(early, on='ticker', how='left').fillna({'pre2023_count': 0})
ticker_stats['pre2023_count'] = ticker_stats['pre2023_count'].astype(int)

print("PER-TICKER SUMMARY:")
print(ticker_stats.sort_values('region').to_string(index=False))


In [ ]:
# ── Ticker volume bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 7))

# Volume
ts_sorted = ticker_stats.sort_values('count', ascending=True)
region_color_map = {'US': '#4CAF50', 'IN': '#2196F3', 'CRYPTO': '#FF5722'}
colors = [region_color_map[r] for r in ts_sorted['region']]

bars = axes[0].barh(ts_sorted['ticker'], ts_sorted['count'],
                    color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel("Number of Articles")
axes[0].set_title("News Volume per Ticker", fontweight='bold')
for bar in bars:
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 str(int(bar.get_width())), va='center', fontsize=7)

from matplotlib.patches import Patch
legend_els = [Patch(color=c, label=r) for r, c in region_color_map.items()]
axes[0].legend(handles=legend_els, loc='lower right')

# Articles per day (news intensity)
ts_sorted2 = ticker_stats.sort_values('articles_per_day', ascending=True)
colors2 = [region_color_map[r] for r in ts_sorted2['region']]
bars2 = axes[1].barh(ts_sorted2['ticker'], ts_sorted2['articles_per_day'],
                     color=colors2, alpha=0.85, edgecolor='white')
axes[1].set_xlabel("Articles per Active Day")
axes[1].set_title("News Intensity per Ticker\n(articles / unique days)", fontweight='bold')
for bar in bars2:
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{bar.get_width():.1f}", va='center', fontsize=7)
axes[1].legend(handles=legend_els, loc='lower right')

plt.suptitle("Ticker-Level News Volume", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("eda_ticker_volume.png", bbox_inches='tight', dpi=150)
plt.show()


## 🔁 5. Duplicate Analysis

In [ ]:
# ── Three types of duplicates ─────────────────────────────────────────────────
dup_title_only      = df_raw[df_raw.duplicated(subset=['title'], keep=False)]
dup_ticker_title    = df_raw[df_raw.duplicated(subset=['ticker','title'], keep=False)]
dup_content_hash    = df_raw[df_raw.duplicated(subset=['content_hash'], keep=False)]

print("DUPLICATE ANALYSIS")
print("=" * 55)
print(f"  Duplicate titles (any ticker)   : {df_raw['title'].duplicated().sum():,} rows")
print(f"  Duplicate (ticker + title)      : {df_raw.duplicated(subset=['ticker','title']).sum():,} rows")
print(f"  Duplicate content_hash          : {df_raw['content_hash'].duplicated().sum():,} rows")
print()
print("  ⚠️  Gap between title-dup and ticker+title dup:")
gap = df_raw['title'].duplicated().sum() - df_raw.duplicated(subset=['ticker','title']).sum()
print(f"     {gap:,} articles have the SAME title but DIFFERENT tickers")
print(f"     → These are cross-ticker news reposts (e.g. CNBC earnings roundup")
print(f"       mentioning both NVDA and MSFT, filed under both tickers)")

print()
print("── Top 10 most duplicated titles ───────────────────────────────")
top_dups = (dup_title_only
    .groupby('title')
    .agg(tickers=('ticker', lambda x: list(x.unique())),
         count=('ticker','count'))
    .sort_values('count', ascending=False)
    .head(10))
for title, row in top_dups.iterrows():
    print(f"  [{row['count']}x] {title[:80]}...")
    print(f"        Tickers: {row['tickers']}")
    print()


In [ ]:
# ── Impact of dedup strategies on class distribution ─────────────────────────
strategies = {
    "No dedup (raw)"             : df_raw,
    "Drop dup (ticker+title)"    : df_raw.drop_duplicates(subset=['ticker','title'], keep='first'),
    "Drop dup (title only)"      : df_raw.drop_duplicates(subset=['title'], keep='first'),
    "Drop dup + pre-2023 filter" : df_raw.drop_duplicates(subset=['ticker','title'], keep='first')
                                         .query("pub_year >= 2023"),
}

print(f"{'Strategy':<38} {'Rows':>8} {'Unique Tickers':>15} {'Rows Removed':>13}")
print("-" * 78)
for name, subset in strategies.items():
    removed = len(df_raw) - len(subset)
    print(f"  {name:<36} {len(subset):>8,} {subset['ticker'].nunique():>15} {removed:>13,}")

print()
print("💡 RECOMMENDATION for Step 2:")
print("   Use 'Drop dup (ticker+title)' → preserves cross-ticker news signal")
print("   AND apply pub_year >= 2023 filter → ensures yfinance can retrieve prices")
print("   Final expected dataset size: ~14,300–14,400 rows")


## 📝 6. Title & Text Quality

In [ ]:
# ── Title length distribution ─────────────────────────────────────────────────
df_raw['title_len']   = df_raw['title'].str.len()
df_raw['summary_len'] = df_raw['summary'].str.len()
# Title ≈ summary check (many summaries are just title + source)
df_raw['title_in_summary'] = df_raw.apply(
    lambda r: str(r['title'])[:50].lower() in str(r['summary']).lower(), axis=1
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Title length histogram
axes[0,0].hist(df_raw['title_len'], bins=60, color='steelblue',
               edgecolor='white', alpha=0.85)
axes[0,0].axvline(df_raw['title_len'].median(), color='red', linestyle='--',
                  label=f"Median={df_raw['title_len'].median():.0f}")
axes[0,0].axvline(200, color='orange', linestyle=':', label='200-char threshold')
axes[0,0].set_title("Title Length Distribution", fontweight='bold')
axes[0,0].set_xlabel("Characters"); axes[0,0].set_ylabel("Count")
axes[0,0].legend()

# Summary length histogram
axes[0,1].hist(df_raw['summary_len'], bins=60, color='coral',
               edgecolor='white', alpha=0.85)
axes[0,1].axvline(df_raw['summary_len'].median(), color='blue', linestyle='--',
                  label=f"Median={df_raw['summary_len'].median():.0f}")
axes[0,1].set_title("Summary Length Distribution", fontweight='bold')
axes[0,1].set_xlabel("Characters"); axes[0,1].set_ylabel("Count")
axes[0,1].legend()

# Title vs summary similarity
sim_counts = df_raw['title_in_summary'].value_counts()
axes[1,0].bar(['Summary ≠ Title\n(unique info)', 'Summary = Title\n(redundant)'],
              sim_counts[[False, True]].values if False in sim_counts.index else [0, sim_counts[True]],
              color=['#4CAF50','#F44336'], edgecolor='white', linewidth=1.5)
axes[1,0].set_title("Is Summary Unique vs Title?", fontweight='bold')
axes[1,0].set_ylabel("Count")
pct_same = df_raw['title_in_summary'].mean() * 100
axes[1,0].text(1, sim_counts.max()*0.6 if hasattr(sim_counts,'max') else 1000,
               f"{pct_same:.1f}%\nredundant", ha='center', fontsize=12, color='white', fontweight='bold')

# Length scatter
sample = df_raw.sample(min(2000, len(df_raw)), random_state=42)
axes[1,1].scatter(sample['title_len'], sample['summary_len'],
                  alpha=0.3, s=10, color='mediumpurple')
axes[1,1].set_xlabel("Title Length"); axes[1,1].set_ylabel("Summary Length")
axes[1,1].set_title("Title vs Summary Length Scatter", fontweight='bold')
corr = df_raw['title_len'].corr(df_raw['summary_len'])
axes[1,1].text(0.05, 0.93, f"Pearson r = {corr:.3f}", transform=axes[1,1].transAxes, fontsize=11)

plt.suptitle("Text Quality Analysis", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("eda_text_quality.png", bbox_inches='tight', dpi=150)
plt.show()

print(f"\nTitle length  — mean: {df_raw['title_len'].mean():.1f}, median: {df_raw['title_len'].median():.1f}")
print(f"Summary length— mean: {df_raw['summary_len'].mean():.1f}, median: {df_raw['summary_len'].median():.1f}")
print(f"Very long titles (>200 chars): {(df_raw['title_len'] > 200).sum()}")
print(f"Redundant summaries (title ⊂ summary): {df_raw['title_in_summary'].sum():,} ({pct_same:.1f}%)")


## 🏛️ 7. Source Credibility

In [ ]:
# ── Source distribution ────────────────────────────────────────────────────────
print(f"Unique source names    : {df_raw['source_name'].nunique()}")
print(f"Unique credibility vals: {df_raw['source_credibility'].nunique()}")
print()
print("Credibility value counts:")
print(df_raw['source_credibility'].value_counts().to_string())
print()

# High-credibility sources
high_cred = df_raw[df_raw['source_credibility'] >= 0.75]
print(f"High-credibility articles (≥0.75): {len(high_cred):,}")
print("Their source names:")
print(high_cred['source_name'].value_counts().head(10))

print()
print("💡 NOTE: Only two credibility levels (0.6 and 0.78) exist.")
print("   The 0.78 sources are likely curated premium outlets.")
print("   We will use this as a binary feature in model training.")


## 💰 8. Price Fetch Planning (Sets Up Step 2)

In [ ]:
# ── Plan the exact date windows we need to fetch from yfinance ────────────────
# For each ticker, we need price data from:
#   min(pub_date) - 5 days   → buffer for weekends/holidays
#   max(pub_date) + 5 days   → to compute 1D, 2D, 3D forward returns

df_recent = df_raw[df_raw['pub_year'] >= 2023].copy()

fetch_plan = df_recent.groupby(['ticker', 'region']).agg(
    article_count = ('title', 'count'),
    fetch_start   = ('pub_date', lambda x: str(min(x))),
    fetch_end     = ('pub_date', lambda x: str(max(x))),
).reset_index()

# Determine yfinance ticker format
def get_yf_ticker(ticker: str, region: str) -> str:
    """Map our ticker to yfinance format."""
    if region == 'CRYPTO':
        return ticker   # BTC-USD, ETH-USD already in yf format
    if ticker.endswith('.NS'):
        return ticker   # Indian NSE tickers
    return ticker       # US tickers (AAPL, MSFT, etc.)

fetch_plan['yf_ticker'] = fetch_plan.apply(
    lambda r: get_yf_ticker(r['ticker'], r['region']), axis=1)

# Identify potential issues
fetch_plan['interval'] = fetch_plan['region'].map({
    'US': '1d',
    'IN': '1d',
    'CRYPTO': '1d',   # crypto trades 24/7, no market hours issue
})

print("PRICE FETCH PLAN")
print("=" * 85)
print(f"{'Ticker':<18} {'Region':<8} {'yf_ticker':<18} {'Articles':>9} {'From':<12} {'To':<12}")
print("=" * 85)
for _, row in fetch_plan.sort_values('region').iterrows():
    print(f"  {row['ticker']:<16} {row['region']:<8} {row['yf_ticker']:<18} "
          f"{row['article_count']:>9,} {row['fetch_start']:<12} {row['fetch_end']:<12}")

print()
print(f"Total tickers to fetch : {len(fetch_plan)}")
print(f"Estimated API calls    : {len(fetch_plan)} (1 per ticker, full range)")
print(f"Typical fetch time     : ~2–5 minutes (yfinance rate limits)")


In [ ]:
# ── Labeling strategy preview ─────────────────────────────────────────────────
# Before Step 3 we need to decide label thresholds.
# Show the expected label distribution under different threshold choices.

import numpy as np
import matplotlib.pyplot as plt

thresholds = [0.5, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 4))

np.random.seed(42)
# Simulate plausible 1D return distribution for financial stocks
sim_returns = np.random.normal(loc=0.02, scale=1.5, size=14000)  # mean ~0%, std ~1.5%

for ax, thresh in zip(axes, thresholds):
    buy  = (sim_returns >  thresh).sum()
    sell = (sim_returns < -thresh).sum()
    hold = len(sim_returns) - buy - sell
    total = len(sim_returns)
    
    values = [sell/total*100, hold/total*100, buy/total*100]
    bars = ax.bar(['SELL','HOLD','BUY'], values,
                  color=['#F44336','#9E9E9E','#4CAF50'], edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    ax.set_title(f"Threshold = ±{thresh}%\nSELL:{sell:,} HOLD:{hold:,} BUY:{buy:,}",
                 fontweight='bold')
    ax.set_ylim(0, 100)
    ax.set_ylabel("% of articles")
    
    imbalance = max(sell, hold, buy) / min(sell, hold, buy + 1)
    color = 'red' if imbalance > 5 else 'orange' if imbalance > 2 else 'green'
    ax.text(0.5, 0.92, f"Imbalance: {imbalance:.1f}x",
            transform=ax.transAxes, ha='center', fontsize=9,
            color=color, fontweight='bold')

plt.suptitle("Simulated Label Distribution at Different Threshold Values\n"
             "(actual distribution determined after yfinance price fetch in Step 2)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("eda_label_simulation.png", bbox_inches='tight', dpi=150)
plt.show()

print()
print("💡 RECOMMENDED THRESHOLD: ±1.0% for US stocks, ±0.75% for Indian stocks")
print("   (Indian market has lower intraday volatility on average)")
print("   Crypto threshold: ±3.0% (much higher volatility)")
print()
print("⚠️  WATCH OUT: 'Hold' class will be majority regardless of threshold.")
print("   → We will address this with SMOTE + class-weighted loss in Step 6.")


## 📋 9. Step 1 Audit Report & Go/No-Go for Step 2

In [ ]:
# ── Final audit summary ────────────────────────────────────────────────────────
df_clean_preview = (df_raw
    .drop_duplicates(subset=['ticker','title'], keep='first')
    .query("pub_year >= 2023")
)

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              STEP 1 AUDIT REPORT — SUMMARY                         ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Raw dataset              : {len(df_raw):>7,} articles                        ║")
print(f"║  After ticker+title dedup : {df_raw.drop_duplicates(subset=['ticker','title']).shape[0]:>7,} articles                        ║")
print(f"║  After 2023+ date filter  : {len(df_clean_preview):>7,} articles                        ║")
print(f"║  Tickers remaining        : {df_clean_preview['ticker'].nunique():>7,}                                ║")
print(f"║  Unique days covered      : {df_clean_preview['pub_date'].nunique():>7,}                                ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  ISSUES FOUND                                                       ║")
print(f"║  1. Labels 100% missing → generate from price reactions ✅          ║")
print(f"║  2. {df_raw['title'].duplicated().sum():,} dup titles ({df_raw.duplicated(subset=['ticker','title']).sum():,} ticker+title) → dedup strategy set ✅  ║")
print(f"║  3. {(df_raw['pub_year'] < 2023).sum()} pre-2023 rows → will be dropped (no price data) ✅         ║")
print(f"║  4. Source credibility only 2 values → binary feature only ✅       ║")
print(f"║  5. ~{df_raw['title_in_summary'].mean()*100:.0f}% summaries redundant with title → use title as primary ✅  ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  PLAN FOR STEP 2                                                    ║")
print("║  • Fetch daily OHLCV via yfinance for all 49 tickers                ║")
print("║  • Compute 1D, 2D, 3D forward close-to-close returns                ║")
print("║  • Handle market hours: US(NYSE), India(NSE), Crypto(24/7)          ║")
print("║  • Label thresholds: US ±1%, India ±0.75%, Crypto ±3%              ║")
print("║  • Merge returns onto news by (ticker, next trading date)           ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  ✅  GO for Step 2 — Price Data Fetch                               ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

# Save the clean preview for Step 2
df_clean_preview.to_csv("step1_clean_preview.csv", index=False)
fetch_plan.to_csv("step1_fetch_plan.csv", index=False)
print()
print("Files saved for Step 2:")
print("  • step1_clean_preview.csv  (deduped + filtered dataset)")
print("  • step1_fetch_plan.csv     (ticker → fetch dates mapping)")
